## Define libraries

In [1]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from helper import RAGHelper
from dotenv import load_dotenv
load_dotenv()
import os
# import pandas as pd
# import re
# import json

# import numpy as np
# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import f1_score

In [3]:
# ingestion
# embedding_model - 2
# chunking_size - 256, 512, 1024
# chunking_overlap - 200, 100, 50
# search_type - 2
# search_k - 3, 5, 7
# vector_database - 3
# retrieval techniques - 2
# reranking
# gen_model - 2



## Define Variables

In [3]:
# Constants

DATASET_SOURCE = 'rungalileo/ragbench'
DATASET_NAME = {"finqa":"earning reports", "tatqa": "financial reports"}
DATA_SPLIT = 'test'
VECTOR_DATABASE = 'chroma'
CHUNKING_SIZES = [256, 512, 1024]
CHUNKING_OVERLAPS = [50, 100, 200]
SEPARATORS = ["\n\n", "\n", " ", ".", ","]
DOMAIN = 'finance'







In [7]:
### Parameters


# embedding
chromadb_folder = "../database"
# gen_model = "llama-3.1-8b-instant"
embedding_model = "BAAI/LLM-Embedder"

# db_name = f"finance_{chunking_size}_{chunking_overlap}"
# persist_directory = f"{chromadb_folder}/{db_name}"
collection_name = embedding_model.replace("/", "_")
embedding_function = HuggingFaceEmbeddings(model=embedding_model)

# search_type = "similarity"
# search_kwargs = {"k":3}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/28.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
def deduplicate_data(data, doc_type):
    data_dict = {}
    for d in data:
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            data_dict[document] = {"docid":[d["id"]]}
    for k in data_dict:
        data_dict[k]["document_type"] = doc_type
        
    return data_dict

## Data Fetching

In [14]:
for k, v in DATASET_NAME.items():
    print(f"Creating embeddings for {k} using {embedding_model}")
    MAX_CHUNKS = 5000

    dataset = load_dataset(DATASET_SOURCE, k, split=DATA_SPLIT)
    dedup = deduplicate_data(dataset, v)
    docs = [
        Document(
            metadata=v, 
            page_content=k
        )
        for k, v in dedup.items()
    ]
    
    for cs, co in zip(CHUNKING_SIZES, CHUNKING_OVERLAPS):
        db_name = f"{DOMAIN}_{cs}_{co}"
        persist_directory = f"{chromadb_folder}/{db_name}"
            
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=co, separators=SEPARATORS)
        docs_chunks = text_splitter.split_documents(docs)
        print(len(docs_chunks))
        
        if len(docs_chunks) > MAX_CHUNKS:
            for i in range(0, len(docs_chunks), MAX_CHUNKS):
                doc_chunk = docs_chunks[i:i+MAX_CHUNKS]
        
                if os.path.exists(persist_directory) and os.listdir(persist_directory):
                    print(f"Loading existing vector database...{db_name}")
                    vector_db = Chroma(
                        collection_name=collection_name,
                        persist_directory=persist_directory, 
                        embedding_function=embedding_function
                    )
                    vector_db.add_documents(doc_chunk)
                else:
                    print(f"Creating new vector database...{db_name}")
                    vector_db = Chroma.from_documents(
                        collection_name=collection_name,
                        documents=doc_chunk, 
                        embedding=embedding_function, 
                        persist_directory=persist_directory
                    )
        else:
            # FIXED: This 'if' statement is now perfectly aligned with the 'else' below it
            if os.path.exists(persist_directory) and os.listdir(persist_directory):
                print(f"Loading existing vector database...{db_name}")
                vector_db = Chroma(
                    collection_name=collection_name,
                    persist_directory=persist_directory, 
                    embedding_function=embedding_function
                )
                vector_db.add_documents(docs_chunks)
            else:
                print(f"Creating new vector database...{db_name}")
                vector_db = Chroma.from_documents(
                    collection_name=collection_name,
                    documents=docs_chunks, 
                    embedding=embedding_function, 
                    persist_directory=persist_directory
                )

print(f"Data load into vector database completed for {embedding_model}")

Creating embeddings for finqa using BAAI/LLM-Embedder
7328
Loading existing vector database...finance_256_50
Loading existing vector database...finance_256_50
3721
Loading existing vector database...finance_512_100
1909
Loading existing vector database...finance_1024_200
Creating embeddings for tatqa using BAAI/LLM-Embedder
2485
Loading existing vector database...finance_256_50
1278
Loading existing vector database...finance_512_100
684
Loading existing vector database...finance_1024_200
Data load into vector database completed for BAAI/LLM-Embedder
